# BasketIQ – Supermarket Transaction Analaysis and Intelligence Platform
---

## Imports

In [3]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

## Load Dataset

In [4]:
df = pd.read_excel("H:\PROJECTS\AIML PROJECTS\BasketIQ\data\Online_Retail.xlsx")
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (541909, 8)


,##,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Basic Info

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   ##           541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


## Missing Values

In [6]:
df.isnull().sum()

##                  0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## Remove Cancelled Orders

In [ ]:
print(df.columns)

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'TotalAmount', 'Hour', 'Day',
       'Month', 'Year', 'Weekday', 'MonthYear'],
      dtype='str')


In [37]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

## Remove Negative Quantities

In [38]:
df = df[df['Quantity'] > 0]

## Remove Missing Customer IDs


In [39]:
df = df.dropna(subset=['CustomerID'])

## Convert InvoiceDate to Datetime

In [40]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

## Feature Engineering (Basic)

In [41]:
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']
df['Hour'] = df['InvoiceDate'].dt.hour
df['Day'] = df['InvoiceDate'].dt.day
df['Month'] = df['InvoiceDate'].dt.month
df['Year'] = df['InvoiceDate'].dt.year
df['Weekday'] = df['InvoiceDate'].dt.day_name()

## Check After Cleaning

In [42]:
print("Cleaned Shape:", df.shape)
df.head()

Cleaned Shape: (397924, 15)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,Hour,Day,Month,Year,Weekday,MonthYear
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,8,1,12,2010,Wednesday,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,8,1,12,2010,Wednesday,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday,2010-12


In [43]:
df.to_csv(r"H:\PROJECTS\AIML PROJECTS\BasketIQ\outputs\cleaned_data.csv", index=False)
print("Cleaned data saved to 'outputs/cleaned_data.csv'")

Cleaned data saved to 'outputs/cleaned_data.csv'


## Load Cleaned Data

In [44]:
df = pd.read_csv("../outputs/cleaned_data.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,Hour,Day,Month,Year,Weekday,MonthYear
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,8,1,12,2010,Wednesday,2010-12
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday,2010-12
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,8,1,12,2010,Wednesday,2010-12
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday,2010-12
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday,2010-12


## Invoice-Level Aggregation
create:
- InvoiceTotal
- ItemCount
- UniqueProducts

In [45]:
invoice_df = df.groupby('InvoiceNo').agg({
    'TotalAmount': 'sum',
    'Quantity': 'sum',
    'StockCode': 'nunique',
    'Hour': 'first',
    'Month': 'first'
}).reset_index()

invoice_df.columns = [
    'InvoiceNo',
    'InvoiceTotal',
    'ItemCount',
    'UniqueProducts',
    'Hour',
    'Month'
]

invoice_df.head()

,InvoiceNo,InvoiceTotal,ItemCount,UniqueProducts,Hour,Month
0,536365,139.12,40,7,8,12
1,536366,22.20,12,2,8,12
2,536367,278.73,83,12,8,12
3,536368,70.05,15,4,8,12
4,536369,17.85,3,1,8,12


## Save Invoice Features

In [46]:
invoice_df.to_csv("../outputs/invoice_features.csv", index=False)
print("Saved invoice_features.csv")

Saved invoice_features.csv


## Customer-Level Aggregation (RFM)
- Reference date = last transaction date + 1 day

In [47]:
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (reference_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalAmount': 'sum'
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

rfm.head()


,CustomerID,Recency,Frequency,Monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


## Add Average Spend Feature

In [48]:
rfm['AvgSpend'] = rfm['Monetary'] / rfm['Frequency']
rfm.head()

,CustomerID,Recency,Frequency,Monetary,AvgSpend
0,12346.0,326,1,77183.60,77183.600000
1,12347.0,2,7,4310.00,615.714286
2,12348.0,75,4,1797.24,449.310000
3,12349.0,19,1,1757.55,1757.550000
4,12350.0,310,1,334.40,334.400000


## Save Customer Features

In [49]:
rfm.to_csv("../outputs/customer_features.csv", index=False)
print("Saved customer_features.csv")

Saved customer_features.csv


## Load Invoice Features

In [ ]:
invoice_df = pd.read_csv("../outputs/invoice_features.csv")
invoice_df.head()

,InvoiceNo,InvoiceTotal,ItemCount,UniqueProducts,Hour,Month
0,536365,139.12,40,7,8,12
1,536366,22.20,12,2,8,12
2,536367,278.73,83,12,8,12
3,536368,70.05,15,4,8,12
4,536369,17.85,3,1,8,12


## Select Features for Anomaly Detection
- InvoiceTotal
- ItemCount
- Hour

In [51]:
features = invoice_df[['InvoiceTotal', 'ItemCount', 'Hour']]
features.head()

,InvoiceTotal,ItemCount,Hour
0,139.12,40,8
1,22.20,12,8
2,278.73,83,8
3,70.05,15,8
4,17.85,3,8


## Scale Features
-  Isolation Forest works better with scaled data.

In [52]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

## Train Isolation Forest

In [53]:
from sklearn.ensemble import IsolationForest
iso = IsolationForest(
    contamination=0.02,
    random_state=42
)
invoice_df['anomaly'] = iso.fit_predict(X_scaled)

- -1 = anomaly
- 1 = normal

## Extract Anomalies

In [54]:
anomalies = invoice_df[invoice_df['anomaly'] == -1]
print("Total anomalies:", anomalies.shape[0])
anomalies.head()

Total anomalies: 371


,InvoiceNo,InvoiceTotal,ItemCount,UniqueProducts,Hour,Month,anomaly
20,536387,3193.92,1440,5,9,12,-1
72,536532,1919.14,1852,73,13,12,-1
209,536783,4076.48,2909,38,15,12,-1
233,536809,1003.20,1824,1,16,12,-1
241,536830,2002.40,4280,2,17,12,-1


## Save Anomaly Results

In [55]:
anomalies.to_csv("../outputs/anomalies.csv", index=False)
print("Saved anomalies.csv")

Saved anomalies.csv


## Save Models (.pkl)

In [56]:
import joblib
joblib.dump(iso, "../models/anomaly_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
print("Models saved")

Models saved


## Load Customer Features

In [13]:
rfm = pd.read_csv("../outputs/customer_features.csv")
rfm.head()

,CustomerID,Recency,Frequency,Monetary,AvgSpend
0,12346.0,326,1,77183.60,77183.600000
1,12347.0,2,7,4310.00,615.714286
2,12348.0,75,4,1797.24,449.310000
3,12349.0,19,1,1757.55,1757.550000
4,12350.0,310,1,334.40,334.400000


## Select Features for Clustering
- Recency
- Frequency
- Monetary
- AvgSpend

In [14]:
X = rfm[['Recency', 'Frequency', 'Monetary', 'AvgSpend']]
X.head()

,Recency,Frequency,Monetary,AvgSpend
0,326,1,77183.60,77183.600000
1,2,7,4310.00,615.714286
2,75,4,1797.24,449.310000
3,19,1,1757.55,1757.550000
4,310,1,334.40,334.400000


## Scale Features

In [15]:
from sklearn.preprocessing import StandardScaler
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X)

## Train KMeans
- 4 clusters

In [16]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['Cluster'] = kmeans.fit_predict(X_scaled)

## Inspect Cluster Summary

In [17]:
rfm.groupby('Cluster').mean()

,CustomerID,Recency,Frequency,Monetary,AvgSpend
Cluster,,,,,
0,15282.777261,41.285006,4.687732,1906.263378,391.716193
1,14396.000000,163.500000,1.500000,122828.050000,80709.925000
2,15355.171271,246.498158,1.581031,523.674485,326.991453
3,15178.826087,6.304348,73.217391,84505.353478,1619.805961


This shows:
- Loyal customers
- High spenders
- At-risk customers

## Save Clustered Customers

In [18]:
rfm.to_csv("../outputs/clusters.csv", index=False)
print("Saved clusters.csv")

Saved clusters.csv


## Save Models

In [19]:
import joblib
joblib.dump(kmeans, "../models/clustering_model.pkl")
joblib.dump(scaler_cluster, "../models/cluster_scaler.pkl")
print("Clustering models saved")

Clustering models saved


## Load Cleaned Data

In [20]:
df = pd.read_csv("../outputs/cleaned_data.csv")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,Hour,Day,Month,Year,Weekday
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,8,1,12,2010,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,8,1,12,2010,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday


## Create Basket Format
Transform:

Invoice → Products

In [22]:
basket = (
    df.groupby(['InvoiceNo', 'Description'])['Quantity']
      .sum()
      .unstack()
      .fillna(0)
)

basket = (basket > 0).astype(int)

basket.head()


Description   4 PURPLE FLOCK DINNER CANDLES   50'S CHRISTMAS GIFT BAG LARGE  \
InvoiceNo                                                                     
536365                                    0                               0   
536366                                    0                               0   
536367                                    0                               0   
536368                                    0                               0   
536369                                    0                               0   

Description   DOLLY GIRL BEAKER   I LOVE LONDON MINI BACKPACK  \
InvoiceNo                                                       
536365                        0                             0   
536366                        0                             0   
536367                        0                             0   
536368                        0                             0   
536369                        0                             0   

Description   I LOVE LONDON MINI RUCKSACK   NINE DRAWER OFFICE TIDY  \
InvoiceNo                                                             
536365                                  0                         0   
536366                                  0                         0   
536367                                  0                         0   
536368                                  0                         0   
536369                                  0                         0   

Description   OVAL WALL MIRROR DIAMANTE    RED SPOT GIFT BAG LARGE  \
InvoiceNo                                                            
536365                                 0                         0   
536366                                 0                         0   
536367                                 0                         0   
536368                                 0                         0   
536369                                 0                         0   

Description   SET 2 TEA TOWELS I LOVE LONDON    SPACEBOY BABY GIFT SET  \
InvoiceNo                                                                
536365                                      0                        0   
536366                                      0                        0   
536367                                      0                        0   
536368                                      0                        0   
536369                                      0                        0   

Description   TOADSTOOL BEDSIDE LIGHT    TRELLIS COAT RACK  \
InvoiceNo                                                    
536365                               0                   0   
536366                               0                   0   
536367                               0                   0   
536368                               0                   0   
536369                               0                   0   

Description  10 COLOUR SPACEBOY PEN  12 COLOURED PARTY BALLOONS  \
InvoiceNo                                                         
536365                            0                           0   
536366                            0                           0   
536367                            0                           0   
536368                            0                           0   
536369                            0                           0   

Description  12 DAISY PEGS IN WOOD BOX  12 EGG HOUSE PAINTED WOOD  \
InvoiceNo                                                           
536365                               0                          0   
536366                               0                          0   
536367                               0                          0   
536368                               0                          0   
536369                               0                          0   

Description  12 HANGING EGGS HAND PAINTED  12 IVORY ROSE PEG PLACE SETTINGS  \
InvoiceNo     

In [23]:
from mlxtend.frequent_patterns import apriori, association_rules
frequent_items = apriori(basket, min_support=0.02, use_colnames=True)
frequent_items.head()

C:\Users\hariv\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\mlxtend\frequent_patterns\fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,support,itemsets
0,0.021688,frozenset({3 STRIPEY MICE FELTCRAFT})
1,0.039167,frozenset({6 RIBBONS RUSTIC CHARM})
2,0.025140,frozenset({60 CAKE CASES VINTAGE CHRISTMAS})
3,0.035445,frozenset({60 TEATIME FAIRY CAKE CASES})
4,0.027028,frozenset({72 SWEETHEART FAIRY CAKE CASES})


## Generate Association Rules

In [24]:
rules = association_rules(
    frequent_items,
    metric="lift",
    min_threshold=1
)
rules = rules.sort_values('lift', ascending=False)
rules.head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
73,frozenset({PINK REGENCY TEACUP AND SAUCER}),"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",0.029996,0.029186,0.021040,0.701439,24.033032,1.0,0.020165,3.251641,0.988027,0.551627,0.692463,0.711163
72,"frozenset({GREEN REGENCY TEACUP AND SAUCER, RO...",frozenset({PINK REGENCY TEACUP AND SAUCER}),0.029186,0.029996,0.021040,0.720887,24.033032,1.0,0.020165,3.475313,0.987204,0.551627,0.712256,0.711163
71,"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.023522,0.037279,0.021040,0.894495,23.994742,1.0,0.020163,9.124923,0.981409,0.529172,0.890410,0.729447
74,frozenset({GREEN REGENCY TEACUP AND SAUCER}),"frozenset({PINK REGENCY TEACUP AND SAUCER, ROS...",0.037279,0.023522,0.021040,0.564399,23.994742,1.0,0.020163,2.241683,0.995433,0.529172,0.553907,0.729447
8,frozenset({PINK REGENCY TEACUP AND SAUCER}),frozenset({GREEN REGENCY TEACUP AND SAUCER}),0.029996,0.037279,0.024817,0.827338,22.193256,1.0,0.023698,5.575760,0.984471,0.584498,0.820652,0.746520


## Save Rules

In [25]:
rules.to_csv("../outputs/rules.csv", index=False)
print("Saved rules.csv")

Saved rules.csv


## Load Cleaned Data

In [26]:
df = pd.read_csv("../outputs/cleaned_data.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,Hour,Day,Month,Year,Weekday
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,8,1,12,2010,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,8,1,12,2010,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday


## Peak Shopping Windows (Hour Analysis)

In [27]:
df = pd.read_csv("../outputs/cleaned_data.csv")
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,Hour,Day,Month,Year,Weekday
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,8,1,12,2010,Wednesday
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,8,1,12,2010,Wednesday
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,8,1,12,2010,Wednesday


## Save Peak Hours

In [29]:
peak_hours = (
    df.groupby('Hour')['InvoiceNo']
      .count()
      .reset_index(name='Transactions')
)

peak_hours.head()

,Hour,Transactions
0,6,1
1,7,379
2,8,8691
3,9,21945
4,10,37999


In [30]:
peak_hours.to_csv("../outputs/peak_hours.csv", index=False)
print("Saved peak_hours.csv")

Saved peak_hours.csv


## Monthly Sales Trends

In [31]:
df['MonthYear'] = df['InvoiceDate'].dt.to_period('M')
monthly_sales = df.groupby('MonthYear')['TotalAmount'].sum().reset_index()
monthly_sales['MonthYear'] = monthly_sales['MonthYear'].astype(str)
monthly_sales.head()

,MonthYear,TotalAmount
0,2010-12,572713.890
1,2011-01,569445.040
2,2011-02,447137.350
3,2011-03,595500.760
4,2011-04,469200.361


## Save Monthly Sales

In [32]:
monthly_sales.to_csv("../outputs/monthly_sales.csv", index=False)
print("Saved monthly_sales.csv")

Saved monthly_sales.csv


## Holiday / Revenue Spike Detection

In [33]:
daily_sales = df.groupby(df['InvoiceDate'].dt.date)['TotalAmount'].sum().reset_index()
threshold = daily_sales['TotalAmount'].mean() + 2 * daily_sales['TotalAmount'].std()
spikes = daily_sales[daily_sales['TotalAmount'] > threshold]
spikes.head()

,InvoiceDate,TotalAmount
32,2011-01-18,87589.11
231,2011-09-15,71993.67
235,2011-09-20,103435.81
248,2011-10-05,74108.43
279,2011-11-10,70513.29


## Save Revenue Spikes

In [34]:
spikes.to_csv("../outputs/revenue_spikes.csv", index=False)
print("Saved revenue_spikes.csv")

Saved revenue_spikes.csv
